In [1]:
import os 
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import psutil, gc
import time 
import json
import pprint

from collections import defaultdict
import random
import numpy as np

In [2]:
import torch 
from torch.nn import CrossEntropyLoss
import torch.nn.functional as F

from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoConfig
from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

### Reward Models

In [3]:
# base_dir
base_dir = '/groups/chichengz/tnn/datasets/'

llm_dir = base_dir + "Qwen2.5-Math-PRM-7B"
# llm_dir = base_dir + "Llama3.1-8B-PRM-Deepseek-Data"
# llm_dir = base_dir + "Skywork-Reward-V2-Llama3.2-3B"
# llm_dir = base_dir + "Skywork-Reward-V2-Llama3.2-8B"

In [4]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    llm_dir,
    trust_remote_code=True
)

# # # Crucial for batching: assign a pad token if missing
# # if tokenizer.pad_token_id is None:
# #     tokenizer.pad_token_id = tokenizer.eos_token_id
# #     tokenizer.pad_token = tokenizer.eos_token

# # Use left-padding if you plan to append generations later, 
# # otherwise right-padding is standard for pure classification/scoring.
# tokenizer.padding_size = 'right'


# Load and Patch the Configuration
config = AutoConfig.from_pretrained(
    llm_dir, 
    trust_remote_code=True
)

# Force the config to have a pad_token_id. 
# We use the tokenizer's eos_token_id as the pad token, which is standard for Qwen.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.pad_token = tokenizer.eos_token
    
config.pad_token_id = tokenizer.pad_token_id

# Load model
llm_tf = AutoModel.from_pretrained(
    llm_dir,
    config=config,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)
llm_tf.eval()
# model_regular.generation_config.pad_token_id = tokenizer.eos_token_id
gc.collect();torch.cuda.empty_cache();
free_memory, total_memory = torch.cuda.mem_get_info(0)
print(f'#--- memory: {(total_memory - free_memory) / (1024**3):.2f} GB')

Loading weights:   0%|          | 0/342 [00:00<?, ?it/s]

Qwen2ForProcessRewardModel LOAD REPORT from: /groups/chichengz/tnn/datasets/Qwen2.5-Math-PRM-7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#--- memory: 13.56 GB


In [5]:
stop

NameError: name 'stop' is not defined

### Generative Models

In [ ]:
# base_dir
base_dir = '/groups/chichengz/tnn/datasets/'

llm_dir = base_dir + "Llama3.2-1B-Instruct"
llm_dir = base_dir + "Llama3.2-3B-Instruct"
llm_dir = base_dir + "Qwen2.5-3B-Instruct"
llm_dir = base_dir + "Qwen2.5-7B-Instruct"


In [ ]:
# Measure the occupied GPU memory using Transformers
# Transformers more closely reflects the actual memory requirements 
# of the LLM, since it allocates memory as needed. 
# In contrast, vLLM reserves a fixed amount of memory upfront for
# its PagedAttention KV cache, which can be larger than the minimum 
# required memory to improve generation speed.
llm_tf = AutoModelForCausalLM.from_pretrained(
    llm_dir,
    device_map="cuda:0",
    trust_remote_code=True,
)
llm_tf.eval()
# model_regular.generation_config.pad_token_id = tokenizer.eos_token_id
gc.collect();torch.cuda.empty_cache();
free_memory, total_memory = torch.cuda.mem_get_info(0)
print(f'#--- memory: {(total_memory - free_memory) / (1024**3):.2f} GB')

In [ ]:
# # Measure the occupied GPU memory using vLLM
# llm_vllm = LLM(
#     model=llm_dir, 
#     tensor_parallel_size=1, 
#     max_model_len=5000,
#     gpu_memory_utilization=0.25,
#     enforce_eager=True,
#     distributed_executor_backend=None,
#     disable_log_stats=True,
#     dtype="float16",
#     seed=0,
# )
# gc.collect();torch.cuda.empty_cache();
# free_memory, total_memory = torch.cuda.mem_get_info(0)
# print(f'#--- memory: {(total_memory - free_memory) / (1024**3):.2f}/{(total_memory) / (1024**3):.2f} GB')
# # print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

In [ ]:
stop